<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_13_extendedPAIRS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 13 — extended PAIRS: unbinned representations and fast frequentist inference

We start from the controlled misspecification in **Exercise 7** and implement three pieces:

1. Learn an event encoder using singletons and pairs, then freeze it to represent full unbinned experiments.
2. Learn the simulator-to-reference response map with an amortized population fit.
3. Train fast estimator and test-statistic heads on cached full-experiment embeddings.

Each piece has a numerical comparison with the direct unbinned calculation. **This notebook stops before distribution-flow training and Neyman calibration.** Agreement with a likelihood teacher is not a coverage statement.

Our starting point is [PAIRS, *It Just Takes Two*](https://arxiv.org/abs/2605.07972): train a mean-pool encoder at set sizes one and two, freeze it, and train an inference head on full-set embeddings. Here we adapt that strategy to a discrete auxiliary posterior, a population response fit, and frequentist regression heads. These extensions are our prototype, not results established by PAIRS. Its sufficiency theorem does not automatically apply to this finite, imperfectly trained representation. [LF2I](https://arxiv.org/abs/2107.03920) motivates the later separation of statistic learning, calibration, and coverage diagnostics.

This first example has **one signal-strength parameter and no nuisance parameters**. Its simplicity lets us test every component before extending the parameter space. No event histogram or $q$ binning is used.

**Prerequisites:** run Exercise 5 to save its PRESEL, reference-flow and ratio checkpoints, and run Exercise 7 through loading the PRESEL state. We reuse those frozen models without retraining them. Use a GPU runtime and the same Google Drive workspace. The `full` configuration retains Exercise 7's nominal event yields; `quick` reduces training and bank sizes, not the event count per experiment.


**This revision:** reuse the completed representation, response and toy caches; compare a signed-root statistic head with a flexible direct-statistic head; train longer with learning-rate reduction; and benchmark the pair posterior against an exact reference calculation. Neither head assumes a quadratic likelihood. The original head is retained as a comparison when its checkpoint exists. The default reuses Rafael's completed run `5f97501b6b87`; on a fresh workspace the notebook builds the prerequisites first.


In [ ]:
# Colab setup: a separate source checkout, the same persistent model workspace.
import os
import sys
import subprocess
from pathlib import Path

USE_DRIVE = True
REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
IN_COLAB = "google.colab" in sys.modules

def run(*args, **kwargs):
    subprocess.run([str(a) for a in args], check=True, **kwargs)

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        ROOT = Path('/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab')
    else:
        ROOT = Path('/content')
    SOURCE_DIR = Path('/content/nsbi_extended_pairs_source')
    if not (SOURCE_DIR / '.git').exists():
        env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
        run('git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
            '--branch', BRANCH, REPO_URL, SOURCE_DIR, env=env)
        run('git', '-C', SOURCE_DIR, 'sparse-checkout', 'set',
            'src', 'workshops/ml4hep_tifr_colab')
    else:
        run('git', '-C', SOURCE_DIR, 'pull', '--ff-only', 'origin', BRANCH)
    TUTORIAL_DIR = SOURCE_DIR / 'workshops/ml4hep_tifr_colab'
    for directory in (SOURCE_DIR / 'src', TUTORIAL_DIR):
        sys.path.insert(0, str(directory))
    run(sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-lightning',
        'onnx', 'onnxruntime', 'onnxscript', 'iminuit', 'mplhep', 'nflows', 'pyarrow')
    WORK_DIR = ROOT / 'nsbi-lhc-toolkit/workshops/ml4hep_tifr'
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
print('Persistent working directory:', Path.cwd())


In [ ]:
import copy
import importlib
import hashlib
import inspect
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
from scipy.optimize import brentq
from IPython.display import display

from nsbi_common_utils.training.utils import load_trained_model
from utils import FEATURES, predict_with_model
from utils_distributions import background_components, signal_components, smearing_parameters
from utils_nf import checkpoint_path, flow_sample_x, load_flow
import utils_extended_pairs as ep
ep = importlib.reload(ep)

FEATURES = list(FEATURES)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 13013
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Training device:', DEVICE)


## Configuration and reuse of the completed run

There is one deformation control, `DEFORMATION_MIXTURE`, used for every event source:

$$r_S^{\rm mis}(x)=(1-\epsilon)r_S^{\rm good}(x)+\epsilon r_B^{\rm good}(x),
\qquad r_B^{\rm mis}(x)=r_B^{\rm good}(x).$$

The physical simulator stays unchanged; we evaluate its events with this same deformed likelihood. `EXPOSURE = 1.0` retains Exercise 7's yields.

`REUSE_RUN` selects the completed data, encoder and response caches. Their saved configuration is checked, and those files are only read. Missing pieces in an existing reused run cause a clear error rather than silently mixing training runs. Set `REUSE_RUN = None` for a deliberate fresh representation/data run, for example after changing the deformation. If the requested run directory is absent, a fresh run is built automatically.

Head settings are separate. New models and reports go into a refinement subdirectory, leaving the original results intact. Changing `HEAD_SETTINGS` or the helper code gives a new refinement directory without regenerating compatible data. Change `REFINEMENT_LABEL` after editing a diagnostic or training function in this notebook.


In [ ]:
MODE = 'full'                  # 'quick' is a structural first pass
DEFORMATION_MIXTURE = 0.05      # match the value you choose in Exercise 7
EXPOSURE = 1.0
RUN_LABEL = 'v1'
REUSE_RUN = '5f97501b6b87'
REFINEMENT_LABEL = 'flexible_heads_v2'
SELECTED_HEAD = 'signed_root'  # fixed choice for the reusable inference function
MU_MAX = 3.0
NU_QUERY_MAX = 5.0

SIZES = {
    'full': dict(bank_train=2_000_000, bank_validation=1_000_000, bank_audit=2_000_000,
                 pairs_train=200_000, pairs_validation=40_000,
                 toys_train=1_500, toys_validation=300, toys_audit=300,
                 pairs_epochs=40, response_epochs=180, heads_epochs=120),
    'quick': dict(bank_train=100_000, bank_validation=50_000, bank_audit=100_000,
                  pairs_train=20_000, pairs_validation=5_000,
                  toys_train=100, toys_validation=30, toys_audit=50,
                  pairs_epochs=8, response_epochs=20, heads_epochs=20),
}[MODE]
EMBED_DIM, WIDTH, N_FRACTIONS = 32, 64, 24
N_NORMALIZATION = 1_000_000
N_QUERY = 10
BATCH_EVENTS = 65_536
assert 0 <= DEFORMATION_MIXTURE <= 1 and EXPOSURE > 0

PRESEL_DIR = Path('models_PRESEL')
FLOW_DIR = Path('models_flows_hybrid_reference_spline16_tail5')
RATIO_DIRS = {'signal': Path('models_Hybrid_SigvsRef_5M_ensemble4'),
              'background': Path('models_Hybrid_BkgvsRef_5M_ensemble4')}
PRESEL_STATE = Path('saved_exercise7_misspecification/exercise5_preselection_state.npz')
# Exercise 7 may have reused this state from Exercise 6 instead of copying it.
STATE_PATHS = [PRESEL_STATE,
    Path('saved_asimov_nis_influence_v2/exercise5_preselection_state.npz'),
    Path('saved_asimov_nis_scan_v1/exercise5_preselection_state.npz')]
PRESEL_STATE = next((p for p in STATE_PATHS if p.exists()), PRESEL_STATE)
CHECKPOINTS = [PRESEL_STATE, PRESEL_DIR / 'model0.onnx',
               PRESEL_DIR / 'model_scaler0.bin',
               checkpoint_path('reference', FLOW_DIR, 'quadratic_spline')]
for directory in RATIO_DIRS.values():
    for member in range(4):
        CHECKPOINTS += [directory / f'model{member}.onnx',
                        directory / f'model_scaler{member}.bin']
missing = [str(p) for p in CHECKPOINTS if not p.exists()]
if missing:
    raise FileNotFoundError('Run Exercises 5 and 7 first. Missing:\n' + '\n'.join(missing))

def file_hash(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

CONFIG = dict(version=1, run_label=RUN_LABEL, mode=MODE, seed=SEED,
    deformation=DEFORMATION_MIXTURE, exposure=EXPOSURE, mu_max=MU_MAX,
    nu_query_max=NU_QUERY_MAX, embed_dim=EMBED_DIM, width=WIDTH,
    n_fractions=N_FRACTIONS, n_normalization=N_NORMALIZATION,
    n_query=N_QUERY, sizes=SIZES, features=FEATURES,
    torch_version=str(torch.__version__),
    files={str(p): file_hash(p) for p in CHECKPOINTS},
    helper_hash=file_hash(ep.__file__),
    source_hashes={name: file_hash(inspect.getfile(function)) for name, function in
        [('simulator', signal_components), ('flow', load_flow), ('ratio', predict_with_model)]})

HEAD_SETTINGS = dict(epochs=400 if MODE == 'full' else 30,
    width=96, batch_size=256, lr=1e-3, patience=70, lr_patience=20,
    lr_factor=0.3, min_lr=1e-5,
    modes=['signed_root', 'direct'], seed=SEED+6,
    estimator_loss='scaled_mse', signed_root_loss='mse', direct_loss='smooth_l1_beta1')

def reusable_settings(config):
    """The settings that determine the frozen data/representation stages."""
    result = copy.deepcopy(config)
    for key in ['helper_hash', 'torch_version', 'version', 'run_label']:
        result.pop(key, None)
    result['sizes'].pop('heads_epochs', None)
    return result

RUN_ROOT = Path('saved_exercise13_extendedPAIRS')
requested = RUN_ROOT / REUSE_RUN if REUSE_RUN else None
REUSING = requested is not None and requested.is_dir()
if REUSING:
    DATA_OUT = requested
    DATA_CONFIG = json.loads((DATA_OUT / 'config.json').read_text())
    if reusable_settings(DATA_CONFIG) != reusable_settings(CONFIG):
        raise ValueError('Settings differ from REUSE_RUN. Restore its settings or set REUSE_RUN=None.')
else:
    if requested is not None:
        print('Requested cache is absent; building a fresh data/representation run.')
    run_id = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
    DATA_OUT = RUN_ROOT / run_id
    DATA_OUT.mkdir(parents=True, exist_ok=True)
    DATA_CONFIG = CONFIG
    (DATA_OUT / 'config.json').write_text(json.dumps(DATA_CONFIG, indent=2))
RUN_ID = DATA_OUT.name
OUT = DATA_OUT  # existing bank/toy construction cells use this name

REFINEMENT_CONFIG = dict(label=REFINEMENT_LABEL, data_run=RUN_ID,
    data_config=DATA_CONFIG, helper_hash=file_hash(ep.__file__),
    torch_version=str(torch.__version__), heads=HEAD_SETTINGS)
refinement_id = hashlib.sha256(json.dumps(REFINEMENT_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
HEAD_OUT = DATA_OUT / 'refinements' / refinement_id
HEAD_OUT.mkdir(parents=True, exist_ok=True)
(HEAD_OUT / 'config.json').write_text(json.dumps(REFINEMENT_CONFIG, indent=2))
print('Frozen data/representation:', DATA_OUT, '(reusing)' if REUSING else '(fresh/cached)')
print('New heads and diagnostics:', HEAD_OUT)

def cached_arrays(name, build, folder=None):
    folder = DATA_OUT if folder is None else folder
    path = folder / f'{name}.npz'
    if path.exists():
        with np.load(path) as saved:
            return {key: saved[key] for key in saved.files}
    if REUSING and folder == DATA_OUT:
        raise FileNotFoundError(f'Missing frozen cache {path}. Set REUSE_RUN=None for a fresh run.')
    values = build()
    np.savez(path, **values)
    return values

def cached_model(name, model, fit, folder=None):
    folder = DATA_OUT if folder is None else folder
    path = folder / f'{name}.pt'
    if path.exists():
        saved = torch.load(path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(saved['state_dict'])
        history = saved['history']
    else:
        if REUSING and folder == DATA_OUT:
            raise FileNotFoundError(f'Missing frozen model {path}. Set REUSE_RUN=None for a fresh run.')
        history = fit()
        torch.save(dict(state_dict=model.state_dict(), history=history), path)
    model.to(DEVICE).eval()
    return history


## Load the frozen model and define the unbinned likelihood

The reference density cancels from the likelihood ratio. For any unbinned experiment $D=\{x_i\}_{i=1}^N$, define

$$q(x)=\frac{\lambda_S r_S^{\rm mis}(x)}{\lambda_B r_B^{\rm mis}(x)},\qquad
\ell_H(\nu;D)-\ell_H(0;D)=-\nu\lambda_S+\sum_{i=1}^N\log(1+\nu q(x_i)).$$

The fit has $\nu\geq0$, with no upper fit cap. The teacher uses individual $q(x_i)$ values and a one-dimensional score root. The encoder receives the original five event features, **not $q$, a fitted parameter, or the true generating parameter**. The exact $q$ calculation is only used to generate teacher labels and validate this prototype.

We normalize the process ratios on a fixed, selected reference-flow sample using the same sample size, seed and procedure as Exercise 7. Its finite normalization is part of the frozen approximate model. This normalization sample is separate from the training, validation and audit event banks.


In [ ]:
def load_ratio_pack(directory, member):
    scaler, proto = load_trained_model(directory / f'model{member}.onnx',
                                       directory / f'model_scaler{member}.bin')
    if isinstance(proto, ort.InferenceSession):
        return scaler, proto
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    session = ort.InferenceSession(proto.SerializeToString(), sess_options=options,
                                    providers=['CPUExecutionProvider'])
    return scaler, session

presel_pack = load_ratio_pack(PRESEL_DIR, 0)
ratio_packs = {name: [load_ratio_pack(directory, k) for k in range(4)]
               for name, directory in RATIO_DIRS.items()}
with np.load(PRESEL_STATE) as state:
    PRESEL_CUT = float(state['ratio_cut'])
    LAM_S = EXPOSURE * float(state['lambda_signal'])
    LAM_B = EXPOSURE * float(state['lambda_background'])
reference_flow = load_flow('reference', model_dir=FLOW_DIR,
    flow_type='quadratic_spline', device=torch.device(DEVICE), expected_features=FEATURES)

def predict(pack, x):
    scaler, session = pack
    frame = pd.DataFrame(np.asarray(x, dtype=np.float32), columns=FEATURES)
    return np.asarray(predict_with_model(frame, scaler=scaler, model=session), dtype=float)

def raw_ratios(x):
    return {name: np.maximum(np.mean([predict(pack, x) for pack in packs], axis=0), 1e-12)
            for name, packs in ratio_packs.items()}

def sample_reference(n, seed):
    torch.manual_seed(seed)
    pieces, kept = [], 0
    while kept < n:
        batch = max(BATCH_EVENTS, min(4 * BATCH_EVENTS, 2 * (n - kept)))
        x = flow_sample_x(reference_flow, batch, batch_size=BATCH_EVENTS)
        selected = x[predict(presel_pack, x) >= PRESEL_CUT]
        pieces.append(selected)
        kept += len(selected)
    return np.concatenate(pieces)[:n].astype(np.float32)

def normalize_reference():
    raw = raw_ratios(sample_reference(N_NORMALIZATION, 12345 + 700))
    return {name: np.asarray(value.mean()) for name, value in raw.items()}

NORMALIZATION = cached_arrays('normalization', normalize_reference)

def model_values(x):
    raw = raw_ratios(x)
    s = raw['signal'] / NORMALIZATION['signal']
    b = raw['background'] / NORMALIZATION['background']
    s = (1 - DEFORMATION_MIXTURE) * s + DEFORMATION_MIXTURE * b
    q = (LAM_S / LAM_B) * s / b
    return q, s, b

print(f'Post-selection yields: signal={LAM_S:.6g}, background={LAM_B:.6g}')
print('Ratio normalization:', NORMALIZATION)


## Independent unbinned event banks

We retain the original features in each bank. Simulator events come from the same latent Gaussian mixtures, detector response and PRESEL as Exercise 7. Reference events come from the frozen flow and carry the deformed process weights.

To make repeated full experiments affordable, we sample events with replacement from these banks. This is a **finite-bank approximation** to each continuous event distribution. Reference process weights are normalized separately for sampling; their means and effective sample sizes below quantify the finite integration sample. The likelihood always uses the fixed normalization above. Small deviations of a reference population fit from its injected value therefore reflect finite-bank integration and fixed normalization-sample error.

Training, validation and final audit use independently generated banks. New Poisson seeds alone would not provide this independence. Increasing the number of toys cannot remove event-bank error; later response plots compare different banks explicitly. We use one bank per process and split, so the configured bank size is not the total number of stored events.


In [ ]:
def simulator_batch(process, n, rng):
    components = signal_components() if process == 'signal' else background_components()
    probabilities = np.array([c[0] for c in components], dtype=float)
    labels = rng.choice(len(components), n, p=probabilities / probabilities.sum())
    x = np.empty((n, len(FEATURES)))
    for k, (_, mean, covariance) in enumerate(components):
        selected = labels == k
        x[selected] = rng.multivariate_normal(mean, covariance, selected.sum())
    scale, resolution = smearing_parameters()
    return (x * np.asarray(scale) + rng.normal(size=x.shape) * resolution).astype(np.float32)

def sample_simulator(process, n, seed):
    rng = np.random.default_rng(seed)
    pieces, kept = [], 0
    while kept < n:
        x = simulator_batch(process, BATCH_EVENTS, rng)
        selected = x[predict(presel_pack, x) >= PRESEL_CUT]
        pieces.append(selected)
        kept += len(selected)
    return np.concatenate(pieces)[:n]

def reference_bank(n, seed):
    x = sample_reference(n, seed)
    q, s, b = model_values(x)
    return dict(x=x, q=q, ps=s / s.sum(), pb=b / b.sum(),
                means=np.array([s.mean(), b.mean()]))

def simulator_bank(process, n, seed):
    x = sample_simulator(process, n, seed)
    return dict(x=x, q=model_values(x)[0])

BANKS, REFERENCE_MEANS, bank_rows = {}, {}, []
for split_index, split in enumerate(['train', 'validation', 'audit']):
    n = SIZES[f'bank_{split}']
    base_seed = SEED + 100 * (split_index + 1)
    ref = cached_arrays(f'{split}_reference', lambda: reference_bank(n, base_seed))
    REFERENCE_MEANS[split] = ref['means']
    BANKS[split] = {'reference': {
        'signal': dict(x=ref['x'], q=ref['q'], p=ref['ps']),
        'background': dict(x=ref['x'], q=ref['q'], p=ref['pb'])}, 'simulator': {}}
    for process_index, process in enumerate(['signal', 'background']):
        sim = cached_arrays(f'{split}_simulator_{process}',
            lambda: simulator_bank(process, n, base_seed + process_index + 1))
        BANKS[split]['simulator'][process] = dict(**sim, p=None)
        p = ref['ps' if process == 'signal' else 'pb']
        bank_rows.append(dict(split=split, process=process, events=n,
            reference_ratio_mean=ref['means'][process_index],
            reference_effective_events=1 / np.sum(p**2)))
    print('Loaded/generated event banks:', split)
display(pd.DataFrame(bank_rows))

def draw_indices(bank, n, rng):
    if bank['p'] is None:
        return rng.integers(len(bank['x']), size=n)
    return rng.choice(len(bank['x']), size=n, replace=True, p=bank['p'])


## 1. Learn a reusable representation from singletons and pairs

At nominal yields, fewer than about one percent of events are signal over much of $\mu\in[0,3]$. A pair drawn only from that design gives a very weak learning signal. We therefore pretrain on a deliberately broader **signal fraction** $f$, sampled uniformly from 24 values between $0.02$ and $0.98$:

$$p_f(x) = f p_S(x) + (1-f)p_B(x).$$

Both events in a pair share the same $f$ and event source. A mean-pool encoder $e_\omega$ gives

$$z(D)=\left(N,\frac{1}{N}\sum_{i=1}^N e_\omega(x_i)\right).$$

The temporary categorical head predicts the posterior probability of the injected fraction class from the pooled embedding, $N$, and the source label. Cross-entropy is the negative log probability of that class. This finite-support auxiliary posterior keeps the implementation small; it is not a continuous posterior flow. These **parameter classes are not event bins**. The source label is used only by this temporary head, so one encoder can learn from both families.

The pretraining fraction is a design variable, not the physical signal-strength prior. Full experiments later use $f=\nu\lambda_S/(\nu\lambda_S+\lambda_B)$ for the reference and the corresponding physical $\mu$ for the simulator. We keep their Poisson counts explicitly.

The held-out cross-entropy should improve over the uniform-prior value $\log K$. This checks that the representation learned something; its usefulness at full event counts must still be tested by the later likelihood comparisons.


In [ ]:
FRACTIONS = np.linspace(0.02, 0.98, N_FRACTIONS)
DOMAINS = ['reference', 'simulator']

def make_pairs(split, count, seed):
    rng = np.random.default_rng(seed)
    label = rng.integers(N_FRACTIONS, size=count)
    n = rng.integers(1, 3, size=count)
    domain = rng.integers(2, size=count)
    # Each row is one experiment. Unused second events are masked by the model.
    process = rng.random((count, 2)) < FRACTIONS[label, None]
    x = np.empty((count, 2, len(FEATURES)), dtype=np.float32)
    for d, source in enumerate(DOMAINS):
        for is_signal, name in [(True, 'signal'), (False, 'background')]:
            rows, slots = np.where((domain[:, None] == d) & (process == is_signal))
            bank = BANKS[split][source][name]
            x[rows, slots] = bank['x'][draw_indices(bank, len(rows), rng)]
    return dict(x=x, n=n, domain=domain, label=label)

pair_train = cached_arrays('pairs_train', lambda: make_pairs('train', SIZES['pairs_train'], SEED+1))
pair_val = cached_arrays('pairs_validation',
    lambda: make_pairs('validation', SIZES['pairs_validation'], SEED+2))
x_scale = pair_train['x'][:, 0]
torch.manual_seed(SEED+3)
encoder = ep.PairEncoder(len(FEATURES), embed_dim=EMBED_DIM, width=WIDTH,
                         x_mean=x_scale.mean(0), x_std=x_scale.std(0))
posterior = ep.PairPosterior(encoder, n_classes=N_FRACTIONS, width=WIDTH)
pair_history = cached_model('pairs', posterior, lambda: ep.train_pairs(
    posterior, pair_train, pair_val, epochs=SIZES['pairs_epochs'],
    batch_size=1024, lr=1e-3, patience=8, seed=SEED+3, device=DEVICE))
encoder.eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)

with torch.no_grad():
    logits = posterior(torch.as_tensor(pair_val['x'], device=DEVICE),
        torch.as_tensor(pair_val['n'], device=DEVICE),
        torch.as_tensor(pair_val['domain'], device=DEVICE))
    pair_loss = torch.nn.functional.cross_entropy(logits,
        torch.as_tensor(pair_val['label'], device=DEVICE), reduction='none').cpu().numpy()
display(pd.DataFrame([dict(source=source, N=n,
    cross_entropy=pair_loss[(pair_val['domain']==d) & (pair_val['n']==n)].mean(),
    prior_cross_entropy=np.log(N_FRACTIONS))
    for d, source in enumerate(DOMAINS) for n in (1, 2)]))


### How close is pair pretraining to the available information?

A small gain over the prior cross-entropy does not by itself imply a poor encoder. Singletons and pairs may contain little information. For the reference family we can calculate the exact posterior over the uniform fraction-class prior:

$$P(f_k\mid D,H)\propto\prod_{i=1}^{N}\left[(1-f_k)+f_k R_H(x_i)\right].$$

Here $R_H=p_{S,H}/p_{B,H}$ includes the **validation bank's process normalizers**, because that is the discrete distribution used to generate these pairs. The background-only factor cancels between classes. The unused second slot of a singleton is masked.

We compare the learned head with this reference posterior. Besides the usual loss on the sampled class label, we report the exact posterior entropy, the neural cross-entropy using the exact posterior probabilities as targets, and their difference $D_{\rm KL}(P_{\rm exact}\Vert P_{\rm neural})$. The latter removes noise from the sampled class label. This benchmark tests the complete encoder/posterior pair; a gap does not isolate which of those two networks is responsible. The simulator is not assigned the misspecified reference posterior as an oracle.

This is a diagnostic only: the existing encoder stays frozen. A significant gap would motivate training with these posterior probabilities, or more informative auxiliary targets, in a separate representation study.


In [ ]:
def reference_pair_check():
    rows = np.flatnonzero(pair_val['domain'] == 0)
    x = pair_val['x'][rows]
    n = pair_val['n'][rows]
    q = model_values(x.reshape(-1, len(FEATURES)))[0].reshape(len(rows), 2)
    ref = BANKS['validation']['reference']
    # pS_j/pB_j = ratio_scale*q_j on this common reference bank.
    ratio_scale = ref['signal']['p'][0] / ref['background']['p'][0] / ref['signal']['q'][0]
    exact = ep.reference_fraction_posterior(q, n, FRACTIONS, ratio_scale)
    with torch.no_grad():
        logits = posterior(torch.as_tensor(x, device=DEVICE),
            torch.as_tensor(n, device=DEVICE),
            torch.zeros(len(rows), dtype=torch.long, device=DEVICE))
        log_neural = logits.log_softmax(-1).cpu().numpy().astype(float)
    return dict(rows=rows, exact=exact, log_neural=log_neural)

oracle = cached_arrays('reference_pair_oracle', reference_pair_check, folder=HEAD_OUT)
oracle_rows, exact = oracle['rows'], oracle['exact']
log_exact, log_neural = np.log(exact), oracle['log_neural']
label = pair_val['label'][oracle_rows]
entropy = -np.sum(exact * log_exact, axis=1)
soft_ce = -np.sum(exact * log_neural, axis=1)
oracle_table = pd.DataFrame([dict(N=n, sets=int((pair_val['n'][oracle_rows]==n).sum()),
    prior_cross_entropy=np.log(N_FRACTIONS),
    neural_label_ce=-log_neural[pair_val['n'][oracle_rows]==n, label[pair_val['n'][oracle_rows]==n]].mean(),
    exact_label_ce=-log_exact[pair_val['n'][oracle_rows]==n, label[pair_val['n'][oracle_rows]==n]].mean(),
    exact_posterior_entropy=entropy[pair_val['n'][oracle_rows]==n].mean(),
    neural_soft_ce=soft_ce[pair_val['n'][oracle_rows]==n].mean(),
    mean_kl=(soft_ce-entropy)[pair_val['n'][oracle_rows]==n].mean()) for n in (1,2)])
display(oracle_table)
oracle_table.to_csv(HEAD_OUT / 'pair_oracle.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, n in zip(axes, (1, 2)):
    i = np.flatnonzero(pair_val['n'][oracle_rows] == n)[0]
    ax.plot(FRACTIONS, exact[i], 'o-', label='Exact reference posterior')
    ax.plot(FRACTIONS, np.exp(log_neural[i]), '--', label='Learned posterior')
    ax.axhline(1/N_FRACTIONS, color='grey', lw=1, label='Uniform prior')
    ax.set(xlabel='Pretraining signal fraction', ylabel='Class probability', title=f'N={n}')
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(HEAD_OUT / 'pair_oracle.png', dpi=150)
plt.show()


### Freeze and cache the event embeddings

The encoder now stays fixed. Encoding each stored event once costs work proportional to the bank size. We then form each experiment by summing its event embeddings and retaining $N$. All sums and means below use float64 so the small fluctuations of a large event set are not lost during aggregation.

The resulting head training no longer backpropagates through thousands of events per experiment. Applying the trained method to a new observed experiment still requires one pass over all its events. An empty experiment has $N=0$ and a zero mean embedding by convention.


The small cache/permutation comparison below is only an absolute diagnostic. A batch-size-dependent encoder difference must be compared with full-experiment fluctuations; the later numerical check does that and measures its effect on inference. We do not loosen a tolerance and interpret that as evidence of negligible inference error.


In [ ]:
encoding_start = time.perf_counter()
for split, sources in BANKS.items():
    for source, processes in sources.items():
        if source == 'reference':
            encoded = cached_arrays(f'{split}_reference_embedding', lambda: dict(
                e=ep.encode_events(encoder, processes['signal']['x'], device=DEVICE)))['e']
            for bank in processes.values():
                bank['e'] = encoded
        else:
            for process, bank in processes.items():
                bank['e'] = cached_arrays(f'{split}_simulator_{process}_embedding',
                    lambda: dict(e=ep.encode_events(encoder, bank['x'], device=DEVICE)))['e']
ENCODING_SECONDS = time.perf_counter() - encoding_start
print(f'Event encoding/cache loading: {ENCODING_SECONDS:.1f} s')

# A small direct check: cached aggregation equals encoding the same raw events.
check_x = BANKS['audit']['simulator']['signal']['x'][:200]
check_e = BANKS['audit']['simulator']['signal']['e'][:200]
direct = ep.encode_events(encoder, check_x[::-1].copy(), device=DEVICE).mean(0, dtype=float)
cached = check_e.mean(0, dtype=float)
print('Permutation/cache maximum difference:', np.max(np.abs(direct-cached)))
# A full-experiment check in standardized units appears after head training.


## 2. Learn the response map with an amortized population fit

The response map is the reference-model parameter that maximizes the **expected** reference log likelihood for simulator experiments:

$$g(\mu)=\arg\max_{\nu\geq0}\mathbb E_{D\sim P_{\rm sim}^{\mu}}[\ell_H(\nu;D)].$$

It is not the mean finite-experiment estimator $\mathbb E[\hat\nu]$. For our additive likelihood, the expectation separates into signal and background event expectations. We train a small network $g_\rho$ directly by minimizing

$$\mathcal L_g=\mathbb E_\mu\left[\lambda_S g_\rho(\mu)
-\mu\lambda_S\mathbb E_{S,\rm sim}\log(1+g_\rho(\mu)q)
-\lambda_B\mathbb E_{B,\rm sim}\log(1+g_\rho(\mu)q)\right].$$

Uniformly sampled $\mu$ values and event minibatches give stochastic gradients. We differentiate only through $g_\rho$, not through the simulator. No grid of fitted response targets is used for training. Independent validation-bank objective values select the checkpoint; the audit bank is used only afterward.

There is one small variance-reduction step: write $\log(1+\nu q)=\nu q+[\log(1+\nu q)-\nu q]$. The full bank gives the linear term's mean exactly; minibatches estimate only the bracketed remainder. This is the same empirical population objective with less noise from the dominant background. Likelihood arithmetic uses float64.

The network enforces nonnegative output but does not force $g(0)=0$. An ideal internally consistent reference model has the identity response $g_H(\nu)=\nu$; we check this numerically below. Unlike $g$, a neural estimator trained on individual experiments learns a finite-sample fitting operation. These are different learning problems.


In [ ]:
torch.manual_seed(SEED+4)
response = ep.ResponseNet(mu_max=MU_MAX, width=32)
sim_train = BANKS['train']['simulator']
sim_val = BANKS['validation']['simulator']
response_history = cached_model('response', response, lambda: ep.train_response(
    response, sim_train['signal']['q'], sim_train['background']['q'], LAM_S, LAM_B,
    val_q_sig=sim_val['signal']['q'], val_q_bkg=sim_val['background']['q'],
    mu_max=MU_MAX, epochs=SIZES['response_epochs'], steps_per_epoch=8,
    mu_batch=32, event_batch=4096, lr=1e-3, patience=25, seed=SEED+4, device=DEVICE))

def response_value(mu):
    with torch.no_grad():
        values = response(torch.as_tensor(np.asarray(mu), dtype=torch.float32, device=DEVICE))
    return values.cpu().numpy().astype(float)

print('Learned g(0), g(1), g(3):', response_value([0, 1, 3]))


### Validate the population fit against direct unbinned score roots

For a chosen bank, the exact scalar score equation is

$$-\lambda_S+\mu\lambda_S\left\langle\frac{q}{1+\nu q}\right\rangle_S
+\lambda_B\left\langle\frac{q}{1+\nu q}\right\rangle_B=0.$$

If its value at zero is nonpositive, the constrained optimum is zero. Otherwise it has a unique positive root when the model is identifiable. We evaluate direct roots only at validation points, after training the response network.

We compare the network with roots on both its training bank and an independent audit bank. Their difference exposes finite-bank integration error. The curvature scale $\sigma_A=I^{-1/2}$ is a useful unit for the response error, not a claim about the simulator's estimator variance under misspecification. The objective gap is $2[\bar\ell(g_{\rm root})-\bar\ell(g_\rho)]$.


In [ ]:
def bank_mean(bank, values):
    return values.mean() if bank['p'] is None else np.dot(bank['p'], values)

def population_score(nu, mu, processes):
    s, b = processes['signal'], processes['background']
    return (-LAM_S + mu * LAM_S * bank_mean(s, s['q'] / (1 + nu*s['q']))
            + LAM_B * bank_mean(b, b['q'] / (1 + nu*b['q'])))

def population_fit(mu, processes):
    if population_score(0, mu, processes) <= 0:
        return 0.0
    upper = max(1.0, mu)
    while population_score(upper, mu, processes) > 0:
        upper *= 2
    return brentq(lambda nu: population_score(nu, mu, processes), 0, upper)

def population_loglik(nu, mu, processes):
    s, b = processes['signal'], processes['background']
    return (-nu*LAM_S + mu*LAM_S*bank_mean(s, np.log1p(nu*s['q']))
            + LAM_B*bank_mean(b, np.log1p(nu*b['q'])))

def population_information(nu, mu, processes):
    s, b = processes['signal'], processes['background']
    return (mu*LAM_S*bank_mean(s, (s['q']/(1+nu*s['q']))**2)
            + LAM_B*bank_mean(b, (b['q']/(1+nu*b['q']))**2))

MU_ANCHORS = np.linspace(0, MU_MAX, 13)
response_rows = []
for mu in MU_ANCHORS:
    learned = float(response_value([mu])[0])
    audit = BANKS['audit']['simulator']
    root = population_fit(mu, audit)
    train_root = population_fit(mu, BANKS['train']['simulator'])
    sigma = population_information(root, mu, audit)**-0.5
    response_rows.append(dict(mu=mu, learned=learned, audit_root=root,
        train_root=train_root, reference_root=population_fit(mu, BANKS['audit']['reference']),
        sigma_A=sigma, error_over_sigma=(learned-root)/sigma,
        bank_shift_over_sigma=(train_root-root)/sigma,
        objective_gap=2*(population_loglik(root, mu, audit)-population_loglik(learned, mu, audit)),
        score_at_learned=population_score(learned, mu, audit)))
response_table = pd.DataFrame(response_rows)
response_table.to_csv(HEAD_OUT / 'response_validation.csv', index=False)
display(response_table)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(MU_ANCHORS, response_table.learned, label='Amortized response')
axes[0].plot(MU_ANCHORS, response_table.audit_root, 'o', label='Audit simulator roots')
axes[0].plot(MU_ANCHORS, response_table.train_root, ':', label='Training-bank roots')
axes[0].plot(MU_ANCHORS, response_table.reference_root, 's', fillstyle='none', label='Audit reference roots')
axes[0].plot(MU_ANCHORS, MU_ANCHORS, '--', color='grey', label='Identity')
axes[0].set(xlabel=r'Physical $\mu$', ylabel=r'Reference parameter $\nu$')
axes[0].legend(fontsize=8)
axes[1].plot(MU_ANCHORS, response_table.error_over_sigma, 'o-', label='Network minus audit root')
axes[1].plot(MU_ANCHORS, response_table.bank_shift_over_sigma, 's--', label='Training minus audit root')
axes[1].axhline(0, color='grey', lw=1)
axes[1].set(xlabel=r'Physical $\mu$', ylabel=r'Difference / $\sigma_A$')
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(HEAD_OUT / 'response_validation.png', dpi=150)
plt.show()
assert response_table.learned.max() < NU_QUERY_MAX, 'Increase NU_QUERY_MAX and rerun for this deformation.'


### Diagnose the reference self-response shift

Reference-bank toys normalize signal and background sampling probabilities separately, whereas the likelihood keeps its original normalization sample. Their small relative normalization difference can visibly shift a high-statistics population fit.

For each independent reference bank, we inspect nested prefixes. If $\bar r_S$ and $\bar r_B$ are the mean frozen ratios on that prefix, then its own process-normalized likelihood ratio would be $q_{\rm same}=q\,\bar r_B/\bar r_S$. The population response computed with this diagnostic ratio closes exactly to the injected value on the same finite bank. Comparing it with the frozen likelihood identifies normalization inconsistency without changing the model used for training or inference.

We reconstruct the original per-event ratios from the cached weights and their stored means. No new simulation is needed. Agreement across increasingly large independent banks tests integration stability; a common offset can also originate in the fixed normalization sample. This check does not replace a future larger-normalization-sample study.


In [ ]:
normalization_rows = []
for split in ['train', 'validation', 'audit']:
    ref = BANKS[split]['reference']
    total = len(ref['signal']['q'])
    s = ref['signal']['p'] * total * REFERENCE_MEANS[split][0]
    b = ref['background']['p'] * total * REFERENCE_MEANS[split][1]
    prefixes = sorted(set([max(2, total//4), max(2, total//2), total]))
    for n in prefixes:
        q = ref['signal']['q'][:n]
        ps, pb = s[:n]/s[:n].sum(), b[:n]/b[:n].sum()
        mean_s, mean_b = s[:n].mean(), b[:n].mean()
        frozen_root = ep.exact_response(q, q, 1., LAM_S, LAM_B, ps, pb)
        same_q = q * mean_b/mean_s
        same_root = ep.exact_response(same_q, same_q, 1., LAM_S, LAM_B, ps, pb)
        normalization_rows.append(dict(split=split, events=n, mean_signal=mean_s,
            mean_background=mean_b, mean_signal_se=s[:n].std(ddof=1)/np.sqrt(n),
            mean_background_se=b[:n].std(ddof=1)/np.sqrt(n),
            frozen_response_at_one=frozen_root, same_bank_response_at_one=same_root))
normalization_table = pd.DataFrame(normalization_rows)
display(normalization_table)
normalization_table.to_csv(HEAD_OUT / 'reference_normalization.csv', index=False)


## 3. Generate full experiments and exact teacher labels

For each physical design point $\mu$, simulator toys use signal mean $\mu\lambda_S$; reference toys use signal mean $g_\rho(\mu)\lambda_S$. Both use background mean $\lambda_B$. The Poisson counts fluctuate independently.

We fit **the same** reference likelihood to both sources and evaluate

$$\hat\nu(D)=\arg\max_{\nu\geq0}\ell_H(\nu;D),\qquad
T_\nu(D)=2[\ell_H(\hat\nu;D)-\ell_H(\nu;D)].$$

Each row stores the embedding, count, teacher estimator and several queried statistic values. Queries include $g_\rho(\mu)$, zero, the exact minimum, and random values in the configured query range. The validation and audit use fresh continuous queries, so they test interpolation rather than memorization of a grid. Whole experiments stay in a single split.

We generate one experiment at a time and discard its event indices after summation and fitting. This avoids storing a giant padded tensor. Nevertheless, drawing and labeling full experiments is still real computation. The acceleration comes afterward, when the cached embeddings are reused for head training and inference. At inference time, neither head receives the source or true $\mu$.


In [ ]:
def draw_experiment(processes, generating_nu, rng):
    total_embedding = np.zeros(EMBED_DIM, dtype=float)
    q_parts = []
    counts = [rng.poisson(generating_nu * LAM_S), rng.poisson(LAM_B)]
    for process, count in zip(['signal', 'background'], counts):
        bank = processes[process]
        indices = draw_indices(bank, count, rng)
        total_embedding += bank['e'][indices].sum(axis=0, dtype=float)
        q_parts.append(bank['q'][indices])
    n = sum(counts)
    z = np.r_[float(n), total_embedding / n if n else total_embedding]
    return z, np.concatenate(q_parts)

def make_toy_table(split, count_per_source, seed):
    rng = np.random.default_rng(seed)
    records = []
    started = time.perf_counter()
    for domain, source in enumerate(DOMAINS):
        mu = rng.uniform(0, MU_MAX, count_per_source)
        # Deliberate endpoint examples, in addition to the continuous design.
        mu[:max(1, count_per_source//10)] = 0
        mu[-max(1, count_per_source//10):] = MU_MAX
        mapped = response_value(mu)
        for i in range(count_per_source):
            generating_nu = mapped[i] if source == 'reference' else mu[i]
            z, q = draw_experiment(BANKS[split][source], generating_nu, rng)
            fitted = ep.exact_fit(q, LAM_S)
            queries = rng.uniform(0, NU_QUERY_MAX, N_QUERY)
            queries[:3] = [mapped[i], 0, fitted]
            statistic = ep.exact_statistic(q, queries, LAM_S, nu_hat=fitted)
            records.append((z, fitted, queries, statistic, mu[i], domain))
            if (i+1) % max(1, count_per_source//5) == 0:
                print(f'{split}: {source} {i+1}/{count_per_source}')
    names = ['z', 'nu_hat', 'nu', 't', 'mu', 'domain']
    result = {name: np.asarray([row[j] for row in records]) for j, name in enumerate(names)}
    result['generation_seconds'] = np.asarray(time.perf_counter()-started)
    return result

TABLES = {}
for k, split in enumerate(['train', 'validation', 'audit']):
    TABLES[split] = cached_arrays(f'{split}_toys', lambda: make_toy_table(
        split, SIZES[f'toys_{split}'], SEED+500+k))
    table = TABLES[split]
    print(split, 'experiments:', len(table['z']),
          'mean N:', table['z'][:, 0].mean(),
          'generation seconds:', float(table['generation_seconds']),
          'fraction of MLEs above query range:', np.mean(table['nu_hat'] > NU_QUERY_MAX))
    assert np.min(table['t']) >= -1e-7
    assert np.max(np.abs(table['t'][:, 2])) < 1e-7


### Train two flexible statistic heads on the same cached experiments

We compare two targets, using identical train/validation splits and network sizes:

* **Signed root:** $r_\nu=\operatorname{sign}(\nu-\hat\nu_{\rm teacher})\sqrt{T_\nu}$. The output is unrestricted and we set $\widehat T_\nu=\widehat r_\nu^2$. Its mean-squared loss is smoother near a single interior likelihood minimum than regression of the unsigned root.
* **Direct statistic:** predict nonnegative $T_\nu$ with a softplus output and use smooth-L1 loss (quadratic for absolute error below one, linear above one). This retains a flexible comparison without allowing the largest tail errors to dominate every update.

Both estimator heads use a ReLU output so exact zeros are possible, with positive initial bias to start training away from a completely inactive output. Their loss is mean-squared estimator error in training-set scale units. Each model still receives only the observed embedding/count and query $\nu$, never the injected parameter or source label.

Neither model assumes a quadratic, convex or single-minimum statistic. The signed-root transformation's smoothness advantage is local to a regular minimum and is not guaranteed with degeneracies; this is why we retain the direct head. The two heads are approximations and their estimator/statistic minima need not be exactly consistent. The exact likelihood teacher in this particular example still exploits the yield-linear model's concavity; an interference example would need a more general teacher fit.

We allow 400 epochs, reduce the learning rate on a validation plateau, and restore the best validation checkpoint. Estimator and statistic losses are recorded separately. The two statistic losses have different units and should not be compared directly; validation RMSE in $T$ is common to both. The original unsigned-root checkpoint is evaluated when available, without retraining it.


In [ ]:
def fit_refined_head(name, train, validation, mode, seed):
    torch.manual_seed(seed)
    model = ep.RefinedHeads(train['z'].shape[1], width=HEAD_SETTINGS['width'], mode=mode)
    history = cached_model(name, model, lambda: ep.train_refined_heads(
        model, train, validation, epochs=HEAD_SETTINGS['epochs'],
        batch_size=HEAD_SETTINGS['batch_size'], lr=HEAD_SETTINGS['lr'],
        patience=HEAD_SETTINGS['patience'], lr_patience=HEAD_SETTINGS['lr_patience'],
        seed=seed, device=DEVICE, lr_factor=HEAD_SETTINGS['lr_factor'],
        min_lr=HEAD_SETTINGS['min_lr']), folder=HEAD_OUT)
    return model, history

MODELS, HISTORIES = {}, {}
for mode in HEAD_SETTINGS['modes']:
    MODELS[mode], HISTORIES[mode] = fit_refined_head(f'heads_{mode}',
        TABLES['train'], TABLES['validation'], mode, HEAD_SETTINGS['seed'])

legacy_path = DATA_OUT / 'heads.pt'
if legacy_path.exists():
    legacy = ep.FastHeads(EMBED_DIM+1, width=96)
    legacy.load_state_dict(torch.load(legacy_path, map_location='cpu', weights_only=True)['state_dict'])
    MODELS['original_unsigned_root'] = legacy.to(DEVICE).eval()

def prediction_metrics(table, predicted_hat, predicted_t):
    dh = predicted_hat-table['nu_hat']
    dt = predicted_t-table['t']
    return dict(estimator_bias=dh.mean(), estimator_rmse=np.sqrt(np.mean(dh**2)),
        statistic_bias=dt[:,0].mean(), statistic_rmse=np.sqrt(np.mean(dt[:,0]**2)),
        scan_statistic_rmse=np.sqrt(np.mean(dt[:,3:]**2)),
        mean_T_at_teacher_minimum=predicted_t[:,2].mean())

validation_rows = []
for name, model in MODELS.items():
    prediction = ep.predict_heads(model, TABLES['validation']['z'], TABLES['validation']['nu'], device=DEVICE)
    validation_rows.append(dict(model=name, **prediction_metrics(TABLES['validation'], *prediction)))
validation_comparison = pd.DataFrame(validation_rows)
display(validation_comparison)
validation_comparison.to_csv(HEAD_OUT / 'validation_comparison.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for col, mode in enumerate(HEAD_SETTINGS['modes']):
    history = pd.DataFrame(HISTORIES[mode])
    for component, color in [('estimator', 'C0'), ('statistic', 'C1')]:
        axes[0,col].plot(history.epoch, history[f'train_{component}'], color=color, ls='--', label=f'Train {component}')
        axes[0,col].plot(history.epoch, history[f'val_{component}'], color=color, label=f'Validation {component}')
    axes[0,col].set(title=mode, xlabel='Epoch', ylabel='Loss component', yscale='log')
    axes[0,col].legend(fontsize=8)
    axes[1,col].plot(history.epoch, history.lr)
    axes[1,col].set(xlabel='Epoch', ylabel='Learning rate', yscale='log')
fig.tight_layout()
fig.savefig(HEAD_OUT / 'head_training.png', dpi=150)
plt.show()

# This audit was inspected in the first run: use it for development comparison.
audit = TABLES['audit']
PREDICTIONS = {name: ep.predict_heads(model, audit['z'], audit['nu'], device=DEVICE)
               for name, model in MODELS.items()}
metric_rows = []
for name, (estimated_hat, estimated_t) in PREDICTIONS.items():
    for d, domain in enumerate(DOMAINS):
        keep = audit['domain']==d
        subset = {key: audit[key][keep] for key in ['nu_hat', 't']}
        metric_rows.append(dict(model=name, source=domain,
            **prediction_metrics(subset, estimated_hat[keep], estimated_t[keep])))
metrics = pd.DataFrame(metric_rows)
display(metrics)
metrics.to_csv(HEAD_OUT / 'head_comparison.csv', index=False)
for name, (estimated_hat, estimated_t) in PREDICTIONS.items():
    np.savez(HEAD_OUT / f'audit_predictions_{name}.npz', nu_hat_fast=estimated_hat, t_fast=estimated_t)

heads = MODELS[SELECTED_HEAD]
hat_fast, t_fast = PREDICTIONS[SELECTED_HEAD]
hat_error, t_error = hat_fast-audit['nu_hat'], t_fast-audit['t']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for d, domain in enumerate(DOMAINS):
    keep = audit['domain']==d
    axes[0].scatter(audit['nu_hat'][keep], hat_fast[keep], s=9, alpha=.5, label=domain)
    axes[1].scatter(audit['t'][keep,0], t_fast[keep,0], s=9, alpha=.5, label=domain)
    axes[2].scatter(audit['mu'][keep], t_error[keep,0], s=9, alpha=.5, label=domain)
for ax in axes[:2]:
    low = min(ax.get_xlim()[0], ax.get_ylim()[0]); high = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([low, high], [low, high], 'k--', lw=1)
    ax.set_xlim(low, high); ax.set_ylim(low, high)
axes[0].set(xlabel=r'Direct $\hat\nu$', ylabel=r'Fast $\hat\nu$')
axes[1].set(xlabel=r'Direct $T_{g(\mu)}$', ylabel=r'Fast $T_{g(\mu)}$')
axes[2].set(xlabel=r'Physical $\mu$', ylabel=r'Fast minus direct $T_{g(\mu)}$')
axes[2].axhline(0, color='grey'); axes[0].legend()
fig.suptitle(f'Selected head: {SELECTED_HEAD}')
fig.tight_layout()
fig.savefig(HEAD_OUT / 'head_validation.png', dpi=150)
plt.show()


### Boundary, tail and interpolation diagnostics

Average regression error can hide the most relevant failures. We therefore inspect the error at the exact likelihood minimum, near several representative statistic levels, and on previously unseen query values. The levels below are merely diagnostic locations; **we are not using them as calibrated cutoffs**.

The exact boundary $\hat\nu=0$ can carry nonzero probability. Report its frequency and the head's behavior separately. A smooth nonnegative approximation need not reproduce this atom exactly, which matters for the later distribution model.


In [ ]:
diagnostics, boundary_rows = [], []
for name, (estimated_hat, estimated_t) in PREDICTIONS.items():
    errors = estimated_t-audit['t']
    for d, domain in enumerate(DOMAINS):
        keep = audit['domain']==d
        diagnostics.append(dict(model=name, source=domain, region='at exact minimum',
            entries=int(keep.sum()), mean_abs_error=np.abs(errors[keep,2]).mean(),
            p95_abs_error=np.quantile(np.abs(errors[keep,2]), .95)))
        for low, high in [(0,1), (1,2.71), (2.71,3.84), (3.84,8), (8,np.inf)]:
            truth, error = audit['t'][keep][:,3:], errors[keep][:,3:]
            selected = (truth>=low) & (truth<high)
            if selected.any():
                diagnostics.append(dict(model=name, source=domain, region=f'{low:g} <= T < {high:g}',
                    entries=int(selected.sum()), mean_abs_error=np.abs(error[selected]).mean(),
                    p95_abs_error=np.quantile(np.abs(error[selected]), .95)))
        low_mu = keep & (audit['mu']<.5)
        boundary_rows.append(dict(model=name, source=domain, toys=int(low_mu.sum()),
            teacher_zero_fraction=np.mean(audit['nu_hat'][low_mu]==0),
            predicted_zero_fraction=np.mean(estimated_hat[low_mu]==0),
            predicted_below_001_fraction=np.mean(estimated_hat[low_mu]<.01),
            estimator_bias= np.mean(estimated_hat[low_mu]-audit['nu_hat'][low_mu])))
display(pd.DataFrame(diagnostics))
display(pd.DataFrame(boundary_rows))
pd.DataFrame(diagnostics).to_csv(HEAD_OUT / 'statistic_regions.csv', index=False)
pd.DataFrame(boundary_rows).to_csv(HEAD_OUT / 'boundary_comparison.csv', index=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
scan_nu = np.linspace(0, NU_QUERY_MAX, 151)
scan_records = []
rng = np.random.default_rng(SEED+900)  # same example experiments as the first run
for row, domain in enumerate(DOMAINS):
    for col, mu in enumerate([0.,1.,3.]):
        generating_nu = response_value([mu])[0] if domain=='reference' else mu
        z, q = draw_experiment(BANKS['audit'][domain], generating_nu, rng)
        fitted = ep.exact_fit(q, LAM_S)
        direct = ep.exact_statistic(q, scan_nu, LAM_S, nu_hat=fitted)
        ax = axes[row,col]
        ax.plot(scan_nu, direct, color='black', label='Direct unbinned')
        for name in HEAD_SETTINGS['modes']:
            predicted_hat, predicted = ep.predict_heads(MODELS[name], z[None], scan_nu[None], device=DEVICE)
            ax.plot(scan_nu, predicted[0], '--', label=name)
            scan_records.append(dict(model=name, source=domain, mu=mu, direct_mle=fitted,
                fast_mle=float(predicted_hat[0]), max_abs_scan_error=float(np.max(np.abs(predicted[0]-direct)))))
        ax.axvline(fitted, color='grey', lw=1)
        ax.set(title=f'{domain}, physical mu={mu:g}', ylabel=r'$T_\nu$', xlabel=r'Query $\nu$', ylim=(0,12))
axes[0,0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(HEAD_OUT / 'unbinned_scans.png', dpi=150)
plt.show()
display(pd.DataFrame(scan_records))
pd.DataFrame(scan_records).to_csv(HEAD_OUT / 'example_scans.csv', index=False)


### What did the learned representation add?

As a simple baseline, repeat the same head training using the observed count $N$ alone. This baseline sees the same experiments and teacher labels. Compare it with the learned embedding on the audit set. Improvement quantifies useful shape information retained by the encoder. If it does not improve, we should investigate representation training before adding calibration flows.

We also inspect estimator means against the population response. They need not coincide at finite event count, especially near the boundary. The response-map definition is unchanged by this comparison.


In [ ]:
count_train = dict(TABLES['train'], z=TABLES['train']['z'][:,:1])
count_val = dict(TABLES['validation'], z=TABLES['validation']['z'][:,:1])
comparison = []
for mode in HEAD_SETTINGS['modes']:
    count_model, count_history = fit_refined_head(f'count_heads_{mode}', count_train, count_val,
                                                 mode, SEED+7)
    count_hat, count_t = ep.predict_heads(count_model, audit['z'][:,:1], audit['nu'], device=DEVICE)
    for d, domain in enumerate(DOMAINS):
        keep = audit['domain']==d
        for label, estimated_hat, estimated_t in [('Count only',count_hat,count_t),
                ('Learned embedding + count',*PREDICTIONS[mode])]:
            comparison.append(dict(model=mode, source=domain, representation=label,
                estimator_rmse=np.sqrt(np.mean((estimated_hat[keep]-audit['nu_hat'][keep])**2)),
                statistic_rmse=np.sqrt(np.mean((estimated_t[keep,0]-audit['t'][keep,0])**2))))
display(pd.DataFrame(comparison))
pd.DataFrame(comparison).to_csv(HEAD_OUT / 'count_baseline.csv', index=False)

endpoint_means = []
for name, (estimated_hat, _) in PREDICTIONS.items():
    for d, domain in enumerate(DOMAINS):
        for mu in [0.,MU_MAX]:
            keep = (audit['domain']==d) & (audit['mu']==mu)
            values = audit['nu_hat'][keep]
            endpoint_means.append(dict(model=name, source=domain, mu=mu,
                g=float(response_value([mu])[0]), mean_direct_estimator=values.mean(),
                mean_fast_estimator=estimated_hat[keep].mean(),
                standard_error_direct=values.std(ddof=1)/np.sqrt(len(values))))
display(pd.DataFrame(endpoint_means))
pd.DataFrame(endpoint_means).to_csv(HEAD_OUT / 'endpoint_means.csv', index=False)


### Does encoding precision matter at full experiment size?

We now compare cached embeddings with fresh encodings of exactly the same events. The first check separates the original bank batch size from a small batch and reversal of that small batch. The second uses full reference and simulator experiments and measures differences in units of the training distribution's embedding standard deviations, followed by their effect on $\hat\nu$ and a statistic scan.

Pooling is float64 throughout. It cannot remove a systematic difference introduced by float32 event encoding. GPU kernels can depend on batch shape, and reduced-precision matrix multiplication can matter. We additionally test fresh encoding with `highest` float32 matmul precision and restore the previous setting afterward. These are diagnostics; a large effect would require a consistent encoding precision and regeneration of embeddings and downstream heads. No absolute tolerance is silently widened.


In [ ]:
bank = BANKS['audit']['simulator']['signal']
block = bank['x'][:min(BATCH_EVENTS, len(bank['x']))]
k = min(200, len(block))
cached_mean = bank['e'][:k].mean(0, dtype=float)
same_batch_mean = ep.encode_events(encoder, block, batch_size=BATCH_EVENTS, device=DEVICE)[:k].mean(0, dtype=float)
small_mean = ep.encode_events(encoder, block[:k], batch_size=BATCH_EVENTS, device=DEVICE).mean(0, dtype=float)
reverse_mean = ep.encode_events(encoder, block[:k][::-1].copy(), batch_size=BATCH_EVENTS, device=DEVICE).mean(0, dtype=float)
display(pd.DataFrame([dict(comparison=label, max_abs_embedding_difference=np.max(np.abs(a-b)))
    for label,a,b in [('cache vs fresh bank-size batch',cached_mean,same_batch_mean),
                      ('bank-size vs small batch',same_batch_mean,small_mean),
                      ('small original vs reversed',small_mean,reverse_mean)]]))

def experiment_for_encoding_check(domain, seed):
    rng = np.random.default_rng(seed)
    generating_nu = response_value([1.])[0] if domain=='reference' else 1.
    raw, cached_events = [], []
    counts = [rng.poisson(generating_nu*LAM_S), rng.poisson(LAM_B)]
    for process,count in zip(['signal','background'], counts):
        bank = BANKS['audit'][domain][process]
        index = draw_indices(bank, count, rng)
        raw.append(bank['x'][index]); cached_events.append(bank['e'][index])
    return np.concatenate(raw), np.concatenate(cached_events)

numerical_rows = []
nu_checks = np.linspace(0, NU_QUERY_MAX, 21)[None]
embedding_scale = heads.z_std.detach().cpu().numpy()[1:]
previous_precision = torch.get_float32_matmul_precision()
for d,domain in enumerate(DOMAINS):
    x, e = experiment_for_encoding_check(domain, SEED+1200+d)
    z_cached = np.r_[len(x), e.mean(0, dtype=float)][None]
    cached_hat, cached_t = ep.predict_heads(heads, z_cached, nu_checks, device=DEVICE)
    fresh = ep.encode_events(encoder, x, batch_size=BATCH_EVENTS, device=DEVICE)
    reversed_e = ep.encode_events(encoder, x[::-1].copy(), batch_size=BATCH_EVENTS, device=DEVICE)
    try:
        torch.set_float32_matmul_precision('highest')
        highest = ep.encode_events(encoder, x, batch_size=BATCH_EVENTS, device=DEVICE)
    finally:
        torch.set_float32_matmul_precision(previous_precision)
    for label,values in [('fresh',fresh), ('fresh reversed',reversed_e), ('fresh highest precision',highest)]:
        z_fresh = np.r_[len(x), values.mean(0, dtype=float)][None]
        h,t = ep.predict_heads(heads, z_fresh, nu_checks, device=DEVICE)
        shift = z_fresh[0,1:]-z_cached[0,1:]
        numerical_rows.append(dict(source=domain, encoding=label, events=len(x),
            max_abs_embedding_difference=np.max(np.abs(shift)),
            max_standardized_embedding_difference=np.max(np.abs(shift/embedding_scale)),
            estimator_difference=float(h[0]-cached_hat[0]),
            max_statistic_difference=np.max(np.abs(t-cached_t))))
numerical_table = pd.DataFrame(numerical_rows)
display(numerical_table)
numerical_table.to_csv(HEAD_OUT / 'embedding_numerics.csv', index=False)


### Timing and a reusable inference function

Timing should separate the one-time preparation from repeated inference. We report cached-head time per experiment and direct fit/statistic time with the same number of queries. The cached timing excludes event encoding, whose total was reported above. For a new dataset, the reusable function below includes the necessary event encoding and count construction.

Its input is already PRESEL-selected reconstructed features in `FEATURES` order. It returns the reference-coordinate estimator and the statistic evaluated at the supplied physical hypotheses through the frozen response map. It does not return confidence intervals or calibrated $p$-values.


In [ ]:
def synchronize():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

# Warm up before timing cached inference.
ep.predict_heads(heads, audit['z'][:10], audit['nu'][:10], device=DEVICE)
synchronize()
start = time.perf_counter()
for _ in range(10):
    ep.predict_heads(heads, audit['z'], audit['nu'], device=DEVICE)
synchronize()
head_seconds = (time.perf_counter()-start) / (10*len(audit['z']))
start = time.perf_counter()
ep.exact_statistic(q, np.linspace(0, NU_QUERY_MAX, N_QUERY), LAM_S)
direct_seconds = time.perf_counter()-start
print(f'Cached head: {head_seconds*1000:.3f} ms/experiment, {N_QUERY} queries')
print(f'Direct fit + {N_QUERY} queries, N={len(q):,}: {direct_seconds*1000:.3f} ms')

def infer_experiment(selected_x, physical_mu):
    event_embedding = ep.encode_events(encoder, np.asarray(selected_x, dtype=np.float32), device=DEVICE)
    n = len(event_embedding)
    mean = event_embedding.mean(0, dtype=float) if n else np.zeros(EMBED_DIM)
    z = np.r_[n, mean][None]
    mu = np.atleast_1d(physical_mu).astype(float)
    if np.any((mu < 0) | (mu > MU_MAX)):
        raise ValueError('Physical hypotheses must lie in the trained range.')
    nu = response_value(mu)
    fitted, statistic = ep.predict_heads(heads, z, nu[None], device=DEVICE)
    return dict(mu=mu, nu_test=nu, nu_hat=float(fitted[0]), t=statistic[0])


manifest = dict(data_run=RUN_ID, refinement=HEAD_OUT.name, features=FEATURES,
    configuration='config.json', data_directory=str(DATA_OUT.resolve()),
    representation=str((DATA_OUT/'pairs.pt').resolve()),
    response=str((DATA_OUT/'response.pt').resolve()), selected_head=SELECTED_HEAD,
    heads={mode: f'heads_{mode}.pt' for mode in HEAD_SETTINGS['modes']},
    tables={split: str((DATA_OUT/f'{split}_toys.npz').resolve()) for split in TABLES},
    statistic='T_nu = 2 [ell_H(nu_hat) - ell_H(nu)], queried at nu=g(mu)',
    calibration_performed=False)
(HEAD_OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Selected head:', SELECTED_HEAD)
print('Saved new models and comparison results in', HEAD_OUT)
print('Frozen data/representation remain in', DATA_OUT)


## Reading this comparison before proceeding

1. Compare the signed-root, direct-statistic and original heads using the same validation/audit experiments. Do the minimum, boundary and tail errors improve? Is estimator error still limiting statistic accuracy?
2. Inspect the separate learning curves. Do losses settle, and did the lower learning rates help? Keep the comparison in common estimator/$T$ units because the training objectives differ.
3. Check the pair-posterior gap against the exact reference benchmark. A small gap suggests little available information was missed by that auxiliary task; it does not prove sufficiency for full-experiment likelihood reconstruction.
4. Compare full-experiment encoding differences in standardized units and their effect on inference. Large changes would require a consistent numerical encoding policy before further training.
5. Inspect normalization and population-response convergence separately from head approximation errors.

These are diagnostic questions, not hard-coded success assertions. No quadratic likelihood approximation is introduced. The signed-root transformation is also an experiment, especially at boundaries or in models with multiple minima.

The current audit has already been inspected and now supports development comparisons. A later coverage study needs fresh simulation after choosing and freezing the representation and statistic. It will train the reference and simulator distribution flows in a separate notebook. This scalar, no-nuisance example does not yet demonstrate high-dimensional profiling or interference fits.

The original run and its models remain available. New heads, learning histories, comparisons, normalization checks and numerical-encoding diagnostics are saved under `refinements/` inside the data run. Changing head settings only retrains those heads; changing the representation or physical model requires a fresh data run.
